## **Model Training**

In [3]:
# train_cnc_02_model.py

import os
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
from sklearn.preprocessing import LabelEncoder


DATA_PATH = "/home/santhosh/AMDA/datasets/CNC_02_dataset.csv"
MODEL_DIR = "/home/santhosh/AMDA/models/CNC_02"
MODEL_PATH = f"{MODEL_DIR}/CNC_02_failure_model.pkl"

FEATURE_COLS = [
    "vibration_mm_s",
    "spindle_current_percent",
    "spindle_temperature_C",
    "acoustic_dba",
    "lubrication_pressure_bar",
    "coolant_temperature_C",
]

CLIP_RANGES = {
    "vibration_mm_s": (0.1, 8.0),
    "spindle_current_percent": (0, 160),
    "spindle_temperature_C": (20, 90),
    "acoustic_dba": (55, 110),
    "lubrication_pressure_bar": (0, 4),
    "coolant_temperature_C": (18, 50),
}


def add_realistic_noise(df, noise_level=0.06):
    np.random.seed(42)
    df = df.copy()

    for col in FEATURE_COLS:
        df[col] += df[col] * np.random.uniform(-noise_level, noise_level, len(df))
        df[col] += np.random.normal(0, df[col].std() * 0.03, len(df))
        low, high = CLIP_RANGES[col]
        df[col] = df[col].clip(low, high)

    return df


def train():
    os.makedirs(MODEL_DIR, exist_ok=True)

    df = pd.read_csv(DATA_PATH)
    df = df.dropna(subset=FEATURE_COLS + ["active_failure"])

    print("Class distribution:")
    print(df["active_failure"].value_counts())

    df = add_realistic_noise(df)

    X = df[FEATURE_COLS]
    y = df["active_failure"]

    encoder = LabelEncoder()
    y_encoded = encoder.fit_transform(y)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y_encoded, test_size=0.3, random_state=42, stratify=y_encoded
    )

    model = RandomForestClassifier(
        n_estimators=400,
        max_depth=12,
        min_samples_split=10,
        min_samples_leaf=5,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1,
    )

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    print("Accuracy:", accuracy_score(y_test, y_pred))
    print("Macro F1:", f1_score(y_test, y_pred, average="macro"))
    print("Weighted F1:", f1_score(y_test, y_pred, average="weighted"))

    print(classification_report(y_test, y_pred, target_names=encoder.classes_))
    print(confusion_matrix(y_test, y_pred))

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    cv_scores = cross_val_score(model, X, y_encoded, cv=cv, scoring="f1_macro", n_jobs=-1)
    print("CV Macro F1:", cv_scores.mean())

    importance = pd.DataFrame({
        "feature": FEATURE_COLS,
        "importance": model.feature_importances_,
    }).sort_values("importance", ascending=False)

    importance.to_csv(f"{MODEL_DIR}/CNC_02_feature_importance.csv", index=False)

    plt.figure(figsize=(8, 5))
    plt.barh(importance["feature"], importance["importance"])
    plt.gca().invert_yaxis()
    plt.title("CNC_02 Feature Importance")
    plt.tight_layout()
    plt.savefig(f"{MODEL_DIR}/CNC_02_feature_importance.png")
    plt.close()

    joblib.dump({
        "model": model,
        "label_encoder": encoder,
        "feature_cols": FEATURE_COLS,
        "classes": list(encoder.classes_),
        "machine_type": "cnc",
        "machine_id": "CNC_02",
    }, MODEL_PATH)

    print("Saved:", MODEL_PATH)


if __name__ == "__main__":
    train()

Class distribution:
active_failure
none                    1022
spindle_misalignment    1004
cutting_overload        1002
coolant_failure          988
chatter                  984
Name: count, dtype: int64
Accuracy: 0.9553333333333334
Macro F1: 0.955312533837002
Weighted F1: 0.9555075024214162
                      precision    recall  f1-score   support

             chatter       0.90      0.96      0.92       295
     coolant_failure       0.96      0.98      0.97       296
    cutting_overload       0.97      0.95      0.96       301
                none       1.00      0.98      0.99       307
spindle_misalignment       0.95      0.92      0.93       301

            accuracy                           0.96      1500
           macro avg       0.96      0.96      0.96      1500
        weighted avg       0.96      0.96      0.96      1500

[[282   1   1   0  11]
 [  1 290   4   1   0]
 [  6   7 285   0   3]
 [  3   3   1 300   0]
 [ 23   0   2   0 276]]
CV Macro F1: 0.9504422946943

## **Testing model against custom test cases**

In [4]:
import pandas as pd
import joblib


MODEL_PATH = "/home/santhosh/AMDA/models/CNC_02/CNC_02_failure_model.pkl"

artifact = joblib.load(MODEL_PATH)

model = artifact["model"]
label_encoder = artifact["label_encoder"]
feature_cols = artifact["feature_cols"]


test_cases = [
    {
        "case": "Healthy baseline",
        "expected": "none",
        "vibration_mm_s": 0.90,
        "spindle_current_percent": 46.0,
        "spindle_temperature_C": 43.0,
        "acoustic_dba": 80.0,
        "lubrication_pressure_bar": 2.00,
        "coolant_temperature_C": 27.0,
    },
    {
        "case": "Healthy with mild load",
        "expected": "none",
        "vibration_mm_s": 1.05,
        "spindle_current_percent": 50.0,
        "spindle_temperature_C": 45.0,
        "acoustic_dba": 82.0,
        "lubrication_pressure_bar": 2.02,
        "coolant_temperature_C": 28.0,
    },
    {
        "case": "Cutting overload",
        "expected": "cutting_overload",
        "vibration_mm_s": 1.8,
        "spindle_current_percent": 82.0,
        "spindle_temperature_C": 63.0,
        "acoustic_dba": 86.0,
        "lubrication_pressure_bar": 2.0,
        "coolant_temperature_C": 33.0,
    },
    {
        "case": "Severe cutting overload",
        "expected": "cutting_overload",
        "vibration_mm_s": 2.3,
        "spindle_current_percent": 94.0,
        "spindle_temperature_C": 70.0,
        "acoustic_dba": 89.0,
        "lubrication_pressure_bar": 1.98,
        "coolant_temperature_C": 36.0,
    },
    {
        "case": "Chatter acoustic dominant",
        "expected": "chatter",
        "vibration_mm_s": 2.6,
        "spindle_current_percent": 54.0,
        "spindle_temperature_C": 50.0,
        "acoustic_dba": 98.0,
        "lubrication_pressure_bar": 2.02,
        "coolant_temperature_C": 28.0,
    },
    {
        "case": "Severe chatter",
        "expected": "chatter",
        "vibration_mm_s": 3.0,
        "spindle_current_percent": 58.0,
        "spindle_temperature_C": 54.0,
        "acoustic_dba": 103.0,
        "lubrication_pressure_bar": 2.01,
        "coolant_temperature_C": 29.0,
    },
    {
        "case": "Coolant failure",
        "expected": "coolant_failure",
        "vibration_mm_s": 1.3,
        "spindle_current_percent": 54.0,
        "spindle_temperature_C": 58.0,
        "acoustic_dba": 84.0,
        "lubrication_pressure_bar": 2.0,
        "coolant_temperature_C": 39.0,
    },
    {
        "case": "Severe coolant failure",
        "expected": "coolant_failure",
        "vibration_mm_s": 1.6,
        "spindle_current_percent": 58.0,
        "spindle_temperature_C": 64.0,
        "acoustic_dba": 87.0,
        "lubrication_pressure_bar": 2.01,
        "coolant_temperature_C": 44.0,
    },
    {
        "case": "Spindle misalignment vibration dominant",
        "expected": "spindle_misalignment",
        "vibration_mm_s": 3.6,
        "spindle_current_percent": 56.0,
        "spindle_temperature_C": 51.0,
        "acoustic_dba": 87.0,
        "lubrication_pressure_bar": 2.0,
        "coolant_temperature_C": 28.0,
    },
    {
        "case": "Severe spindle misalignment",
        "expected": "spindle_misalignment",
        "vibration_mm_s": 4.4,
        "spindle_current_percent": 62.0,
        "spindle_temperature_C": 56.0,
        "acoustic_dba": 90.0,
        "lubrication_pressure_bar": 2.02,
        "coolant_temperature_C": 29.0,
    },
    {
        "case": "Borderline overload vs misalignment",
        "expected": "cutting_overload",
        "vibration_mm_s": 2.5,
        "spindle_current_percent": 73.0,
        "spindle_temperature_C": 59.0,
        "acoustic_dba": 88.0,
        "lubrication_pressure_bar": 2.0,
        "coolant_temperature_C": 32.0,
    },
    {
        "case": "Borderline chatter vs misalignment",
        "expected": "chatter",
        "vibration_mm_s": 3.1,
        "spindle_current_percent": 57.0,
        "spindle_temperature_C": 52.0,
        "acoustic_dba": 94.0,
        "lubrication_pressure_bar": 2.02,
        "coolant_temperature_C": 29.0,
    },
]


df_test = pd.DataFrame(test_cases)

X_test = df_test[feature_cols]

pred_encoded = model.predict(X_test)
pred_labels = label_encoder.inverse_transform(pred_encoded)
probabilities = model.predict_proba(X_test)

df_test["predicted"] = pred_labels
df_test["confidence"] = probabilities.max(axis=1).round(4)
df_test["correct"] = df_test["expected"] == df_test["predicted"]

top2_indices = probabilities.argsort(axis=1)[:, -2:][:, ::-1]

df_test["top_1"] = [
    label_encoder.classes_[idxs[0]] for idxs in top2_indices
]

df_test["top_1_confidence"] = [
    round(probabilities[i][idxs[0]], 4)
    for i, idxs in enumerate(top2_indices)
]

df_test["top_2"] = [
    label_encoder.classes_[idxs[1]] for idxs in top2_indices
]

df_test["top_2_confidence"] = [
    round(probabilities[i][idxs[1]], 4)
    for i, idxs in enumerate(top2_indices)
]

print("\nCNC_02 Scenario Test Results:")
print(
    df_test[
        [
            "case",
            "expected",
            "predicted",
            "confidence",
            "top_1",
            "top_1_confidence",
            "top_2",
            "top_2_confidence",
            "correct",
        ]
    ]
)

accuracy = df_test["correct"].mean()
print(f"\nScenario Test Accuracy: {accuracy:.2%}")

output_path = "/home/santhosh/AMDA/models/CNC_02/CNC_02_manual_scenario_tests.csv"
df_test.to_csv(output_path, index=False)

print("\nSaved test results to:")
print(output_path)


CNC_02 Scenario Test Results:
                                       case              expected  \
0                          Healthy baseline                  none   
1                    Healthy with mild load                  none   
2                          Cutting overload      cutting_overload   
3                   Severe cutting overload      cutting_overload   
4                 Chatter acoustic dominant               chatter   
5                            Severe chatter               chatter   
6                           Coolant failure       coolant_failure   
7                    Severe coolant failure       coolant_failure   
8   Spindle misalignment vibration dominant  spindle_misalignment   
9               Severe spindle misalignment  spindle_misalignment   
10      Borderline overload vs misalignment      cutting_overload   
11       Borderline chatter vs misalignment               chatter   

               predicted  confidence                 top_1  top_1_confi